# Feature Engineering

This section creates new business-oriented features that enhance analytical capabilities and support downstream exploratory analysis.

In [1]:
# Load Clean Dataset


import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/Cleaned/cleaned_dataco.csv",
    parse_dates=[
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ],
    encoding="latin1"
)

In [2]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Region,Order State,Order Status,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Southeast Asia,Java Occidental,COMPLETE,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,South Asia,RajastÃ¡n,PENDING,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,South Asia,RajastÃ¡n,CLOSED,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Oceania,Queensland,COMPLETE,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Oceania,Queensland,PENDING_PAYMENT,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


In [3]:
# Dataset Shape
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Rows    : 180519
Columns : 45


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Type                           180519 non-null  object        
 1   Days for shipping (real)       180519 non-null  int64         
 2   Days for shipment (scheduled)  180519 non-null  int64         
 3   Benefit per order              180519 non-null  float64       
 4   Sales per customer             180519 non-null  float64       
 5   Delivery Status                180519 non-null  object        
 6   Late_delivery_risk             180519 non-null  int64         
 7   Category Id                    180519 non-null  int64         
 8   Category Name                  180519 non-null  object        
 9   Customer City                  180519 non-null  object        
 10  Customer Country               180519 non-null  object        
 11  

In [5]:
# Date Features


df["Order Year"] = df["order date (DateOrders)"].dt.year

df["Order Month"] = df["order date (DateOrders)"].dt.month

df["Order Month Name"] = df["order date (DateOrders)"].dt.month_name()

df["Order Quarter"] = df["order date (DateOrders)"].dt.quarter

df["Order Day"] = df["order date (DateOrders)"].dt.day

df["Order Weekday"] = df["order date (DateOrders)"].dt.day_name()

In [6]:
df["Weekend Order"] = np.where(
    df["Order Weekday"].isin(["Saturday", "Sunday"]),
    "Yes",
    "No"
)

In [7]:
df[[
    "Order Year",
    "Order Month",
    "Order Month Name",
    "Order Quarter",
    "Order Weekday",
    "Weekend Order"
]].head()

,Order Year,Order Month,Order Month Name,Order Quarter,Order Weekday,Weekend Order
0,2018,1,January,1,Wednesday,No
1,2018,1,January,1,Saturday,Yes
2,2018,1,January,1,Saturday,Yes
3,2018,1,January,1,Saturday,Yes
4,2018,1,January,1,Saturday,Yes


In [8]:
# Shipping Features


df["Shipping Delay"] = (
    df["Days for shipping (real)"] -
    df["Days for shipment (scheduled)"]
)

In [9]:
df["Delivery Performance"] = np.where(
    df["Shipping Delay"] > 0,
    "Delayed",
    "On Time"
)

In [10]:
df["Early Delivery"] = np.where(
    df["Shipping Delay"] < 0,
    "Yes",
    "No"
)

In [11]:
df[[
    "Shipping Delay",
    "Delivery Performance",
    "Early Delivery"
]].head()

,Shipping Delay,Delivery Performance,Early Delivery
0,-1,On Time,Yes
1,1,Delayed,No
2,0,On Time,No
3,-1,On Time,Yes
4,-2,On Time,Yes


In [12]:
# Profit Features


df["Profit Margin (%)"] = (
    df["Order Item Profit Ratio"] * 100
)

In [13]:
median_profit = df["Order Profit Per Order"].median()

df["High Profit Order"] = np.where(
    df["Order Profit Per Order"] >= median_profit,
    "High",
    "Low"
)

In [14]:
df[[
    "Profit Margin (%)",
    "High Profit Order"
]].head()

,Profit Margin (%),High Profit Order
0,28.999999,High
1,-80.000001,Low
2,-80.000001,Low
3,8.000000,Low
4,44.999999,High


In [15]:
#Sales feature


df["Sales Bucket"] = pd.cut(
    df["Sales"],
    bins=[0,100,300,600,1000,10000],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [16]:
median_sales = df["Sales"].median()

df["High Value Order"] = np.where(
    df["Sales"] >= median_sales,
    "Yes",
    "No"
)

In [17]:
df[[
    "Sales",
    "Sales Bucket",
    "High Value Order"
]].head()

,Sales,Sales Bucket,High Value Order
0,327.75,Medium,Yes
1,327.75,Medium,Yes
2,327.75,Medium,Yes
3,327.75,Medium,Yes
4,327.75,Medium,Yes


In [18]:
# Discount Feature

df["Discount Applied"] = np.where(
    df["Order Item Discount"] > 0,
    "Yes",
    "No"
)

In [19]:
df["Heavy Discount"] = np.where(
    df["Order Item Discount Rate"] >= 0.20,
    "Yes",
    "No"
)

In [20]:
df[[
    "Discount Applied",
    "Heavy Discount"
]].head()

,Discount Applied,Heavy Discount
0,Yes,No
1,Yes,No
2,Yes,No
3,Yes,No
4,Yes,No


In [21]:
# Shipping Efficiency

df["Shipping Efficiency (%)"] = (
    df["Days for shipment (scheduled)"] /
    df["Days for shipping (real)"]
) * 100

In [22]:
# Customer lifetime revenue

df["Customer Lifetime Revenue"] = (
    df.groupby("Customer Id")["Sales"]
      .transform("sum")
)

In [23]:
# Customer AVG Order Value

df["Customer Average Order Value"] = (
    df.groupby("Customer Id")["Sales"]
      .transform("mean")
)

In [24]:
#Order Size

df["Order Size"] = pd.cut(
    df["Order Item Quantity"],
    bins=[0,1,3,5,100],
    labels=[
        "Single",
        "Small",
        "Medium",
        "Bulk"
    ]
)

In [25]:
#Discount Category

df["Discount Category"] = pd.cut(
    df["Order Item Discount Rate"],
    bins=[-0.01,0,0.10,0.20,1],
    labels=[
        "No Discount",
        "Low",
        "Medium",
        "High"
    ]
)

In [26]:
#Loss Order

df["Loss Order"] = np.where(
    df["Order Profit Per Order"] < 0,
    "Yes",
    "No"
)

In [27]:
#Order Type

df["Order Type"] = np.where(
    df["Customer Country"] == df["Order Country"],
    "Domestic",
    "International"
)

In [28]:
df.drop(columns=["Early Delivery", 
                    "Discount Applied",
                    "High Profit Order",
                    "Order Day"],inplace=True) 

In [29]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Sales Bucket,High Value Order,Heavy Discount,Shipping Efficiency (%),Customer Lifetime Revenue,Customer Average Order Value,Order Size,Discount Category,Loss Order,Order Type
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Medium,Yes,No,133.333333,327.75,327.75,Single,Low,No,International
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,Medium,Yes,No,80.000000,327.75,327.75,Single,Low,Yes,International
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,Medium,Yes,No,100.000000,327.75,327.75,Single,Low,Yes,International
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Medium,Yes,No,133.333333,327.75,327.75,Single,Low,No,International
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Medium,Yes,No,200.000000,327.75,327.75,Single,Low,No,International


In [30]:
df.columns.to_list()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Id',
 'Customer Segment',
 'Customer State',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Card Id',
 'Product Category Id',
 'Product Name',
 'Product Price',
 'Product Status',
 'shipping date (DateOrders)',
 'Shipping Mode',
 'Order Year',
 'Order Month',
 'Order Month Name',
 'Order Quarter',
 'Order Weekda

In [31]:
df[["Customer Id", "Order Customer Id"]].head()

,Customer Id,Order Customer Id
0,20755,20755
1,19492,19492
2,19491,19491
3,19490,19490
4,19489,19489


In [32]:
df["Order Customer Id"].head()

0    20755
1    19492
2    19491
3    19490
4    19489
Name: Order Customer Id, dtype: int64

In [33]:
df.drop(
    columns=[
        "Heavy Discount",
        "High Value Order"
    ],
    inplace=True
)

In [34]:
engineered_features = [
    "Order Year",
    "Order Month",
    "Order Month Name",
    "Order Quarter",
    "Order Weekday",
    "Weekend Order",
    "Shipping Delay",
    "Delivery Performance",
    "Shipping Efficiency (%)",
    "Profit Margin (%)",
    "Sales Bucket",
    "Customer Lifetime Revenue",
    "Customer Average Order Value",
    "Order Size",
    "Discount Category",
    "Loss Order",
    "Order Type"
]

df[engineered_features].head()

,Order Year,Order Month,Order Month Name,Order Quarter,Order Weekday,Weekend Order,Shipping Delay,Delivery Performance,Shipping Efficiency (%),Profit Margin (%),Sales Bucket,Customer Lifetime Revenue,Customer Average Order Value,Order Size,Discount Category,Loss Order,Order Type
0,2018,1,January,1,Wednesday,No,-1,On Time,133.333333,28.999999,Medium,327.75,327.75,Single,Low,No,International
1,2018,1,January,1,Saturday,Yes,1,Delayed,80.000000,-80.000001,Medium,327.75,327.75,Single,Low,Yes,International
2,2018,1,January,1,Saturday,Yes,0,On Time,100.000000,-80.000001,Medium,327.75,327.75,Single,Low,Yes,International
3,2018,1,January,1,Saturday,Yes,-1,On Time,133.333333,8.000000,Medium,327.75,327.75,Single,Low,No,International
4,2018,1,January,1,Saturday,Yes,-2,On Time,200.000000,44.999999,Medium,327.75,327.75,Single,Low,No,International


In [35]:
df[engineered_features].isnull().sum()

Order Year                         0
Order Month                        0
Order Month Name                   0
Order Quarter                      0
Order Weekday                      0
Weekend Order                      0
Shipping Delay                     0
Delivery Performance               0
Shipping Efficiency (%)         5080
Profit Margin (%)                  0
Sales Bucket                       0
Customer Lifetime Revenue          0
Customer Average Order Value       0
Order Size                         0
Discount Category                  0
Loss Order                         0
Order Type                         0
dtype: int64

In [36]:
df[df["Shipping Efficiency (%)"].isnull()][
    [
        "Days for shipping (real)",
        "Days for shipment (scheduled)"
    ]
].head(20)

,Days for shipping (real),Days for shipment (scheduled)
19,0,0
20,0,0
387,0,0
388,0,0
389,0,0
1115,0,0
1197,0,0
1279,0,0
1280,0,0
2571,0,0


In [37]:
df[[
    "Sales",
    "Sales per customer",
    "Order Item Total"
]].head(20)

,Sales,Sales per customer,Order Item Total
0,327.75,314.640015,314.640015
1,327.75,311.359985,311.359985
2,327.75,309.720001,309.720001
3,327.75,304.809998,304.809998
4,327.75,298.250000,298.250000
5,327.75,294.980011,294.980011
6,327.75,288.420013,288.420013
7,327.75,285.140015,285.140015
8,327.75,278.589996,278.589996
9,327.75,275.309998,275.309998


In [38]:
df[[
    "Sales",
    "Sales per customer",
    "Order Item Total"
]].describe()

,Sales,Sales per customer,Order Item Total
count,180519.000000,180519.000000,180519.000000
mean,203.772096,183.107609,183.107609
std,132.273077,120.043670,120.043670
min,9.990000,7.490000,7.490000
25%,119.980003,104.379997,104.379997
50%,199.919998,163.990005,163.990005
75%,299.950012,247.399994,247.399994
max,1999.989990,1939.989990,1939.989990


In [39]:
df[[
    "Sales",
    "Sales per customer",
    "Order Item Total"
]].corr(numeric_only=True)

,Sales,Sales per customer,Order Item Total
Sales,1.000000,0.989744,0.989744
Sales per customer,0.989744,1.000000,1.000000
Order Item Total,0.989744,1.000000,1.000000


In [40]:
df[[
    "Sales",
    "Order Item Discount",
    "Order Item Total"
]].head(20)

,Sales,Order Item Discount,Order Item Total
0,327.75,13.110000,314.640015
1,327.75,16.389999,311.359985
2,327.75,18.030001,309.720001
3,327.75,22.940001,304.809998
4,327.75,29.500000,298.250000
5,327.75,32.779999,294.980011
6,327.75,39.330002,288.420013
7,327.75,42.610001,285.140015
8,327.75,49.160000,278.589996
9,327.75,52.439999,275.309998


In [41]:
(
    df["Sales"] -
    df["Order Item Discount"]
).head(20)

0     314.640000
1     311.360001
2     309.719999
3     304.809999
4     298.250000
5     294.970001
6     288.419998
7     285.139999
8     278.590000
9     275.310001
10    272.029999
11    268.750000
12    262.199997
13    245.809998
14    327.750000
15    324.470000
16    321.190000
17    317.920000
18    314.640000
19    311.360001
dtype: float64

In [42]:
(
    df["Sales per customer"] ==
    df["Order Item Total"]
).value_counts()

True    180519
Name: count, dtype: int64

In [43]:
df.drop(columns=["Sales per customer"],inplace=True)

In [44]:
df[[
    "Benefit per order",
    "Order Profit Per Order",
    "Order Item Profit Ratio"
]].head(20)

,Benefit per order,Order Profit Per Order,Order Item Profit Ratio
0,91.250000,91.250000,0.29
1,-249.089996,-249.089996,-0.80
2,-247.779999,-247.779999,-0.80
3,22.860001,22.860001,0.08
4,134.210007,134.210007,0.45
5,18.580000,18.580000,0.06
6,95.180000,95.180000,0.33
7,68.430000,68.430000,0.24
8,133.720001,133.720001,0.48
9,132.149994,132.149994,0.48


In [45]:
df[[
    "Benefit per order",
    "Order Profit Per Order",
    "Order Item Profit Ratio"
]].corr()

,Benefit per order,Order Profit Per Order,Order Item Profit Ratio
Benefit per order,1.000000,1.000000,0.823689
Order Profit Per Order,1.000000,1.000000,0.823689
Order Item Profit Ratio,0.823689,0.823689,1.000000


In [46]:
df[[
    "Benefit per order",
    "Order Profit Per Order",
    "Order Item Profit Ratio"
]].value_counts()

Benefit per order  Order Profit Per Order  Order Item Profit Ratio
0.000000           0.000000                0.00                       1177
143.990005         143.990005              0.48                        133
52.410000          52.410000               0.48                         87
96.000000          96.000000               0.48                         78
56.779999          56.779999               0.48                         77
                                                                      ... 
2.760000           2.760000                0.01                          1
                                           0.03                          1
                                           0.11                          1
                                           0.14                          1
911.799988         911.799988              0.47                          1
Name: count, Length: 52814, dtype: int64

In [47]:
(df["Benefit per order"] == df["Order Profit Per Order"]).value_counts()

True    180519
Name: count, dtype: int64

In [48]:
df.drop(columns=["Benefit per order"], inplace=True)

In [49]:
# Category Audit
df.groupby("Category Id")["Category Name"].nunique().value_counts()

Category Name
1    51
Name: count, dtype: int64

In [50]:
# Department Audit
df.groupby("Department Id")["Department Name"].nunique().value_counts()

Department Name
1    11
Name: count, dtype: int64

In [51]:
# Product Audit
df.groupby("Product Card Id")["Product Name"].nunique().value_counts()

Product Name
1    118
Name: count, dtype: int64

In [55]:
df.to_csv("../data/Engineered/Featured_Engineering.csv", index=False)

In [53]:
# KPI Validation

gross_revenue = df["Sales"].sum()

net_revenue = df["Order Item Total"].sum()

total_profit = df["Order Profit Per Order"].sum()

total_orders = df["Order Id"].nunique()

total_customers = df["Customer Id"].nunique()

total_quantity = df["Order Item Quantity"].sum()

average_order_value = net_revenue / total_orders

average_delivery_delay = df["Shipping Delay"].mean()

late_orders = df.loc[
    df["Delivery Performance"] == "Delayed",
    "Order Id"
].nunique()

late_delivery_rate = late_orders / total_orders

overall_profit_margin = total_profit / net_revenue

In [54]:
kpi_validation = pd.Series({
    "Gross Revenue": gross_revenue,
    "Net Revenue": net_revenue,
    "Total Profit": total_profit,
    "Total Orders": total_orders,
    "Total Customers": total_customers,
    "Total Quantity Sold": total_quantity,
    "Average Order Value": average_order_value,
    "Average Delivery Delay": average_delivery_delay,
    "Late Delivery Rate": late_delivery_rate,
    "Overall Profit Margin": overall_profit_margin
})

kpi_validation

Gross Revenue             3.678474e+07
Net Revenue               3.305440e+07
Total Profit              3.966903e+06
Total Orders              6.575200e+04
Total Customers           2.065200e+04
Total Quantity Sold       3.840790e+05
Average Order Value       5.027133e+02
Average Delivery Delay    5.658075e-01
Late Delivery Rate        5.733362e-01
Overall Profit Margin     1.200113e-01
dtype: float64